In [1]:
import pandas as pd
import great_expectations as gx
from great_expectations import expectations as gxe

E0000 00:00:1773642491.609762 23982740 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773642491.609941 23982740 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1773642491.609943 23982740 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773642491.609944 23982740 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1773642491.609951 23982740 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


## Data quality check and testing for __fact_orders__
- Check for null values in critical columns such as order_id, customer_id, and order_status.
- Validate data types of columns to ensure they match expected formats (e.g.order_purchase_timestamp should be a timestamp). 
- Check for duplicate records based on order_id.


In [2]:
order_data = pd.read_csv("../data/fact_order.csv")

In [3]:
context = gx.get_context()

data_source_name = "order_dataframe"
data_source = context.data_sources.add_pandas(name=data_source_name)

# create asset
data_asset_name = "order_asset"
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

# create batch
batch_definition_name = "order_dataframe_batch"
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

In [4]:
# get the data via batch
batch_parameters = {"dataframe": order_data}

new_batch = batch_definition.get_batch(batch_parameters=batch_parameters)

In [6]:
# Define expectations for the order data
# For example, we can check that the order_id, customer_id, and order_status columns do not contain null values.
not_null_expectations = [
    gxe.ExpectColumnValuesToNotBeNull(column="order_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="customer_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="order_status"),
]

not_null_expectations

[ExpectColumnValuesToNotBeNull(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_id', mostly=1, row_condition=None, condition_parser=None),
 ExpectColumnValuesToNotBeNull(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='customer_id', mostly=1, row_condition=None, condition_parser=None),
 ExpectColumnValuesToNotBeNull(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_status', mostly=1, row_condition=None, condition_parser=None)]

In [7]:
# Create an expectation suite and add the expectations to it
# An expectation suite is a collection of expectations that can be run against a batch of data.
suite = gx.ExpectationSuite(name="order_not_null_suite")
for expectation in not_null_expectations:
    suite.add_expectation(expectation)

# Add the expectation suite to the context
context.suites.add(suite) 

validation_definition = gx.ValidationDefinition(
    name="order_validation_definition",
    data=batch_definition,
    suite=suite
)

# Run the validation definition and print the results
validation_result = validation_definition.run(batch_parameters=batch_parameters)
print(validation_result)

Calculating Metrics:   0%|          | 0/18 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062841.031170Z",
      "pandas_data_fingerprint": "d842c6ca0b2a379231d8d667636c2ece"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "order_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "dde3eca6-4172-4c9c-941e-fc989c8bab9e",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:28:41.107130+08:00"
    },
    "validation_time": "2026-03-16T06:28:41.107130+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message"

In [8]:
# Define an expectation to check that the order_purchase_timestamp column follows a specific datetime format.
timestamp_expectation = gxe.ExpectColumnValuesToMatchStrftimeFormat(
    column="order_purchase_timestamp",
    strftime_format="%Y-%m-%d %H:%M:%S UTC",
)

# Create an expectation suite for the timestamp expectation
timestamp_suite = gx.ExpectationSuite(name="order_timestamp_suite")
timestamp_suite.add_expectation(timestamp_expectation)

# Add the expectation suite to the context
context.suites.add(timestamp_suite)

# Create a validation definition for the timestamp expectation suite and run it against the batch of data.
timestamp_validation_definition = gx.ValidationDefinition(
    name="order_timestamp_validation_definition",
    data=batch_definition,
    suite=timestamp_suite,
)

# Run the validation and print the results
timestamp_validation_result = timestamp_validation_definition.run(
    batch_parameters=batch_parameters
)
print(timestamp_validation_result)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062848.314638Z",
      "pandas_data_fingerprint": "d842c6ca0b2a379231d8d667636c2ece"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "order_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "dfd627fe-819b-49c0-abd1-74405db11916",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:28:48.581826+08:00"
    },
    "validation_time": "2026-03-16T06:28:48.581826+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message"

In [9]:
# Check that order_id has no duplicates in fact_order
order_id_unique_expectation = gxe.ExpectColumnValuesToBeUnique(column="order_id")

order_id_unique_suite = gx.ExpectationSuite(name="order_id_unique_suite")
order_id_unique_suite.add_expectation(order_id_unique_expectation)
context.suites.add(order_id_unique_suite)

order_id_unique_validation_definition = gx.ValidationDefinition(
    name="order_id_unique_validation_definition",
    data=batch_definition,
    suite=order_id_unique_suite,
)

order_id_unique_validation_result = order_id_unique_validation_definition.run(
    batch_parameters=batch_parameters
)
print(order_id_unique_validation_result)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062915.767600Z",
      "pandas_data_fingerprint": "d842c6ca0b2a379231d8d667636c2ece"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "order_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "434379ca-a0d2-452b-b134-3f9cf7e18529",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:29:15.854548+08:00"
    },
    "validation_time": "2026-03-16T06:29:15.854548+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message"

## Data quality check and testing for __dim_customers__
- Check for null values in critical columns such as customer_id and customer_unique_id.
- Check for duplicate records based on customer_id or customer_unique_id.  

In [10]:
# Load dim_customers data
customer_data = pd.read_csv("../data/dim_customers.csv")

# Create asset and batch definition for customer data
customer_asset_name = "customer_asset"
customer_asset = data_source.add_dataframe_asset(name=customer_asset_name)

customer_batch_definition_name = "customer_dataframe_batch"
customer_batch_definition = customer_asset.add_batch_definition_whole_dataframe(
    customer_batch_definition_name
)

customer_batch_parameters = {"dataframe": customer_data}

# Define not-null expectations for required columns
customer_not_null_expectations = [
    gxe.ExpectColumnValuesToNotBeNull(column="customer_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="customer_unique_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="customer_zipcode"),
    gxe.ExpectColumnValuesToNotBeNull(column="customer_city"),
    gxe.ExpectColumnValuesToNotBeNull(column="customer_state"),
]

# Build suite
customer_suite = gx.ExpectationSuite(name="customer_not_null_suite")
for exp in customer_not_null_expectations:
    customer_suite.add_expectation(exp)

context.suites.add(customer_suite)

# Validate
customer_validation_definition = gx.ValidationDefinition(
    name="customer_validation_definition",
    data=customer_batch_definition,
    suite=customer_suite,
)

customer_validation_result = customer_validation_definition.run(
    batch_parameters=customer_batch_parameters
)
print(customer_validation_result)

Calculating Metrics:   0%|          | 0/28 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062920.452674Z",
      "pandas_data_fingerprint": "6d339f15886e7d0e49079d41287b9e33"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "customer_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "13b43583-deb1-44a5-93fb-77e19f137d1b",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:29:20.509568+08:00"
    },
    "validation_time": "2026-03-16T06:29:20.509568+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_messa

In [11]:
# Check that customer_id has no duplicates in dim_customers
# customer_id is the primary key of dim_customers, so it should be unique.
customer_id_unique_expectation = gxe.ExpectColumnValuesToBeUnique(
    column="customer_id"
)

customer_id_unique_suite = gx.ExpectationSuite(
    name="customer_id_suite"
)
customer_id_unique_suite.add_expectation(customer_id_unique_expectation)
context.suites.add(customer_id_unique_suite)

customer_id_validation_definition = gx.ValidationDefinition(
    name="customer_id_validation_definition",
    data=customer_batch_definition,
    suite=customer_id_unique_suite,
)

customer_id_validation_result = (
    customer_id_validation_definition.run(
        batch_parameters=customer_batch_parameters
    )
)

print(customer_id_validation_result)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062923.661439Z",
      "pandas_data_fingerprint": "6d339f15886e7d0e49079d41287b9e33"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "customer_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "fcfb64ac-60e8-4ace-ab08-4a4bc19c3951",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:29:23.770951+08:00"
    },
    "validation_time": "2026-03-16T06:29:23.770951+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_messa

## Data quality check and testing for __dim_products__
- Check for null values in critical columns such as product_id and product_category_name.
- Check for duplicate records based on product_id.

In [12]:
# Load dim_products data
product_data = pd.read_csv("../data/dim_products.csv")

# Create asset and batch definition for product data
product_asset_name = "product_asset"
product_asset = data_source.add_dataframe_asset(name=product_asset_name)

product_batch_definition_name = "product_dataframe_batch"
product_batch_definition = product_asset.add_batch_definition_whole_dataframe(
    product_batch_definition_name
)

product_batch_parameters = {"dataframe": product_data}

# Non-null checks for required columns
product_not_null_expectations = [
    gxe.ExpectColumnValuesToNotBeNull(column="product_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="product_category_name"),
]

product_not_null_suite = gx.ExpectationSuite(name="product_not_null_suite")
for exp in product_not_null_expectations:
    product_not_null_suite.add_expectation(exp)

context.suites.add(product_not_null_suite)

product_not_null_validation_definition = gx.ValidationDefinition(
    name="product_not_null_validation_definition",
    data=product_batch_definition,
    suite=product_not_null_suite,
)

product_not_null_validation_result = product_not_null_validation_definition.run(
    batch_parameters=product_batch_parameters
)
print(product_not_null_validation_result)

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062928.003410Z",
      "pandas_data_fingerprint": "8dcfe4c37d40ae164a735c50b7b250e0"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "product_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "59f4a6dc-b84b-4d77-9719-74542189e0b8",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:29:28.021010+08:00"
    },
    "validation_time": "2026-03-16T06:29:28.021010+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_messag

In [13]:
# Duplicate check for product_id (must be unique)
product_id_unique_expectation = gxe.ExpectColumnValuesToBeUnique(column="product_id")

product_id_unique_suite = gx.ExpectationSuite(name="product_id_unique_suite")
product_id_unique_suite.add_expectation(product_id_unique_expectation)
context.suites.add(product_id_unique_suite)

product_id_unique_validation_definition = gx.ValidationDefinition(
    name="product_id_unique_validation_definition",
    data=product_batch_definition,
    suite=product_id_unique_suite,
)

product_id_unique_validation_result = product_id_unique_validation_definition.run(
    batch_parameters=product_batch_parameters
)
print(product_id_unique_validation_result)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062937.166139Z",
      "pandas_data_fingerprint": "8dcfe4c37d40ae164a735c50b7b250e0"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "product_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "4e2558af-2575-47c9-8bf6-b8e8fe288206",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:29:37.191067+08:00"
    },
    "validation_time": "2026-03-16T06:29:37.191067+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_messag

## Data quality check and testing for __dim_sellers__
- Check for null values in critical columns such as seller_id, seller_city and seller_state.
- Check for duplicate records based on seller_id.

In [14]:
# Load dim_sellers data
seller_data = pd.read_csv("../data/dim_sellers.csv")

# Create asset and batch definition for seller data
seller_asset_name = "seller_asset"
seller_asset = data_source.add_dataframe_asset(name=seller_asset_name)

seller_batch_definition_name = "seller_dataframe_batch"
seller_batch_definition = seller_asset.add_batch_definition_whole_dataframe(
    seller_batch_definition_name
)

seller_batch_parameters = {"dataframe": seller_data}

# Non-null checks for required columns
seller_not_null_expectations = [
    gxe.ExpectColumnValuesToNotBeNull(column="seller_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="seller_city"),
    gxe.ExpectColumnValuesToNotBeNull(column="seller_state"),
]

seller_not_null_suite = gx.ExpectationSuite(name="seller_not_null_suite")
for exp in seller_not_null_expectations:
    seller_not_null_suite.add_expectation(exp)

context.suites.add(seller_not_null_suite)

seller_not_null_validation_definition = gx.ValidationDefinition(
    name="seller_not_null_validation_definition",
    data=seller_batch_definition,
    suite=seller_not_null_suite,
)

seller_not_null_validation_result = seller_not_null_validation_definition.run(
    batch_parameters=seller_batch_parameters
)
print(seller_not_null_validation_result)

Calculating Metrics:   0%|          | 0/18 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062941.916395Z",
      "pandas_data_fingerprint": "eb854967562f2b76016c8fa3d4ba53ad"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "seller_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "556abdc0-74c8-4049-af5d-22a5b559d2a6",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:29:41.932315+08:00"
    },
    "validation_time": "2026-03-16T06:29:41.932315+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message

In [15]:
# Check duplicate records for seller_id in dim_sellers
seller_id_unique_expectation = gxe.ExpectColumnValuesToBeUnique(column="seller_id")

seller_id_unique_suite = gx.ExpectationSuite(name="seller_id_unique_suite")
seller_id_unique_suite.add_expectation(seller_id_unique_expectation)
context.suites.add(seller_id_unique_suite)

seller_id_unique_validation_definition = gx.ValidationDefinition(
    name="seller_id_unique_validation_definition",
    data=seller_batch_definition,
    suite=seller_id_unique_suite,
)

seller_id_unique_validation_result = seller_id_unique_validation_definition.run(
    batch_parameters=seller_batch_parameters
)
print(seller_id_unique_validation_result)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T062955.763582Z",
      "pandas_data_fingerprint": "eb854967562f2b76016c8fa3d4ba53ad"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "seller_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "a0308a74-992e-4335-8c46-a6c65524ea67",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:29:55.776910+08:00"
    },
    "validation_time": "2026-03-16T06:29:55.776910+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message

## Data quality check and testing for __dim_order_item__
- Check for null values in critical columns such as order_id, product_id, and seller_id.
- Check for duplicate records based on order_id. Note: *This check is not applicable as one order can have multiple items as stated in order_line_id columns.*
- Check for price that are negative or zero, which may indicate data entry errors.
- Check for freight_value that are negative, which may indicate data entry errors.
- Check for total_value that are negative or zero, which may indicate data entry errors.

In [16]:
# Load dim_order_item data
order_item_data = pd.read_csv("../data/dim_order_item.csv")

# Create asset and batch definition for dim_order_item
order_item_asset_name = "order_item_asset"
order_item_asset = data_source.add_dataframe_asset(name=order_item_asset_name)

order_item_batch_definition_name = "order_item_dataframe_batch"
order_item_batch_definition = order_item_asset.add_batch_definition_whole_dataframe(
    order_item_batch_definition_name
)

order_item_batch_parameters = {"dataframe": order_item_data}

# Non-null checks for required columns
order_item_not_null_expectations = [
    gxe.ExpectColumnValuesToNotBeNull(column="order_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="product_id"),
    gxe.ExpectColumnValuesToNotBeNull(column="seller_id"),
]

order_item_not_null_suite = gx.ExpectationSuite(name="order_item_not_null_suite")
for exp in order_item_not_null_expectations:
    order_item_not_null_suite.add_expectation(exp)

context.suites.add(order_item_not_null_suite)

order_item_not_null_validation_definition = gx.ValidationDefinition(
    name="order_item_not_null_validation_definition",
    data=order_item_batch_definition,
    suite=order_item_not_null_suite,
)

order_item_not_null_validation_result = order_item_not_null_validation_definition.run(
    batch_parameters=order_item_batch_parameters
)
print(order_item_not_null_validation_result)

Calculating Metrics:   0%|          | 0/18 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T063003.185440Z",
      "pandas_data_fingerprint": "2b41d78034ea844359aff298d6022df9"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "order_item_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "ad97a2d1-bfbb-4d81-a748-179e6b2e5f1e",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:30:03.232515+08:00"
    },
    "validation_time": "2026-03-16T06:30:03.232515+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_mes

In [17]:
# Check duplicate records for order_id in dim_order_item
order_item_order_id_unique_expectation = gxe.ExpectColumnValuesToBeUnique(
    column="order_id"
)

order_item_order_id_unique_suite = gx.ExpectationSuite(
    name="order_item_order_id_unique_suite"
)
order_item_order_id_unique_suite.add_expectation(order_item_order_id_unique_expectation)
context.suites.add(order_item_order_id_unique_suite)

order_item_order_id_unique_validation_definition = gx.ValidationDefinition(
    name="order_item_order_id_unique_validation_definition",
    data=order_item_batch_definition,
    suite=order_item_order_id_unique_suite,
)

order_item_order_id_unique_validation_result = (
    order_item_order_id_unique_validation_definition.run(
        batch_parameters=order_item_batch_parameters
    )
)
print(order_item_order_id_unique_validation_result)

# Optional quick summary with pandas
duplicate_order_id_count = order_item_data.duplicated(subset=["order_id"]).sum()
print(f"Duplicate order_id rows in dim_order_item: {duplicate_order_id_count}")

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": false,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T063011.047176Z",
      "pandas_data_fingerprint": "2b41d78034ea844359aff298d6022df9"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "order_item_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "c162ff8d-8b39-47a5-a8f8-dd7198373403",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:30:11.117081+08:00"
    },
    "validation_time": "2026-03-16T06:30:11.117081+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_me

### Lesson learned:
- there are quite a lot of duplicated order_id. After investigation, this is expected as one order can have multiple items as stated in order_line_id columns. Hence this quality check is not applicable for dim_order_items.

In [18]:
# Check dim_order_item for strictly positive price and total_value
order_item_positive_value_expectations = [
    gxe.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0,
        strict_min=True,   # price > 0
    ),
    gxe.ExpectColumnValuesToBeBetween(
        column="total_value",
        min_value=0,
        strict_min=True,   # total_value > 0
    ),
]

order_item_positive_value_suite = gx.ExpectationSuite(
    name="order_item_positive_value_suite"
)
for exp in order_item_positive_value_expectations:
    order_item_positive_value_suite.add_expectation(exp)

context.suites.add(order_item_positive_value_suite)

order_item_positive_value_validation_definition = gx.ValidationDefinition(
    name="order_item_positive_value_validation_definition",
    data=order_item_batch_definition,
    suite=order_item_positive_value_suite,
)

order_item_positive_value_validation_result = (
    order_item_positive_value_validation_definition.run(
        batch_parameters=order_item_batch_parameters
    )
)
print(order_item_positive_value_validation_result)

# Optional quick pandas summary
invalid_price_count = (order_item_data["price"] <= 0).sum()
invalid_total_value_count = (order_item_data["total_value"] <= 0).sum()
print(f"Rows with price <= 0: {invalid_price_count}")
print(f"Rows with total_value <= 0: {invalid_total_value_count}")

Calculating Metrics:   0%|          | 0/17 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T063020.677664Z",
      "pandas_data_fingerprint": "2b41d78034ea844359aff298d6022df9"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "order_item_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "5a92beec-2cb2-4d09-b3d4-cc862da56fad",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:30:20.762049+08:00"
    },
    "validation_time": "2026-03-16T06:30:20.762049+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_mes

In [19]:
# Check dim_order_item for non-negative freight_value (freight_value >= 0)
order_item_freight_non_negative_expectation = gxe.ExpectColumnValuesToBeBetween(
	column="freight_value",
	min_value=0,
	strict_min=False,
)

order_item_freight_non_negative_suite = gx.ExpectationSuite(
	name="order_item_freight_non_negative_suite"
)
order_item_freight_non_negative_suite.add_expectation(
	order_item_freight_non_negative_expectation
)
context.suites.add(order_item_freight_non_negative_suite)

order_item_freight_non_negative_validation_definition = gx.ValidationDefinition(
	name="order_item_freight_non_negative_validation_definition",
	data=order_item_batch_definition,
	suite=order_item_freight_non_negative_suite,
)

order_item_freight_non_negative_validation_result = (
	order_item_freight_non_negative_validation_definition.run(
		batch_parameters=order_item_batch_parameters
	)
)
print(order_item_freight_non_negative_validation_result)

# Optional quick pandas summary
negative_freight_count = (order_item_data["freight_value"] < 0).sum()
print(f"Rows with freight_value < 0: {negative_freight_count}")

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": true,
  "id": null,
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_spec": {
      "batch_data": "PandasDataFrame"
    },
    "batch_markers": {
      "ge_load_time": "20260316T063023.830668Z",
      "pandas_data_fingerprint": "2b41d78034ea844359aff298d6022df9"
    },
    "active_batch_definition": {
      "datasource_name": "order_dataframe",
      "data_connector_name": "fluent",
      "data_asset_name": "order_item_asset",
      "batch_identifiers": {
        "dataframe": "<DATAFRAME>"
      }
    },
    "validation_id": "aaae34f1-4050-4abc-a2f3-0d971794aad0",
    "checkpoint_id": null,
    "run_id": {
      "run_name": null,
      "run_time": "2026-03-16T14:30:23.890391+08:00"
    },
    "validation_time": "2026-03-16T06:30:23.890391+00:00",
    "batch_parameters": {
      "dataframe": "<DATAFRAME>"
    }
  },
  "results": [
    {
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_mes